Motivation: Commuting in London can be hit or miss. The TfL app sometimes isn't quick enough to detect delays and service disruption, as they have to confirm everything. Hopefully I make my commute less painful.

In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

# Example API Key:
# API_KEY: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

Function to get TFL's line status. This clearly has limitations: there will be a delay in reporting a change to line status, and change in status on other parts of the line will show here.

In [2]:
def get_line_status(line_id):
    url = f"https://api.tfl.gov.uk/Line/{line_id}/Status"
    params = {"app_key": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

status = get_line_status("district")
for line in status:
    for s in line["lineStatuses"]:
        print(s["statusSeverityDescription"])

Good Service


Function to get arrivals for given stations. Direction keyword is used for ease later.

In [3]:
def find_arrivals(ids, stopPointId, direction="all"):
    url=f"https://api.tfl.gov.uk/Line/{ids}/Arrivals/{stopPointId}?direction={direction}"
    params = {"app_key": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

for stop in ["940GZZLUWIP", "940GZZLUSFS", "940GZZLUEPY", "940GZZLUPYB", "940GZZLUPSG", "940GZZLUFBY"]:  # Wimbledon Park to Fulham Broadway
    arrivals = find_arrivals("district", stop, direction="outbound")
    for tube in arrivals:
        print(tube["naptanId"][-3:], tube["vehicleId"], tube["platformName"], tube["towards"], tube["expectedArrival"], tube["timeToStation"], sep=", ")


WIP, 075, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:12:12Z, 136
SFS, 022, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:10:13Z, 16
SFS, 075, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:14:13Z, 256
EPY, 022, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:13:12Z, 194
EPY, 075, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:16:42Z, 404
PYB, 022, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:14:43Z, 285
PYB, 073, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:10:12Z, 14
PYB, 075, Eastbound - Platform 1, Edgware Road, 2026-07-21T16:18:43Z, 525
PSG, 022, Eastbound - Platform 2, Edgware Road, 2026-07-21T16:17:43Z, 465
PSG, 073, Eastbound - Platform 2, Edgware Road, 2026-07-21T16:12:43Z, 165
PSG, 075, Eastbound - Platform 2, Edgware Road, 2026-07-21T16:21:43Z, 705
FBY, 022, Eastbound - Platform 2, Edgware Road, 2026-07-21T16:19:42Z, 583
FBY, 046, Eastbound - Platform 2, Edgware Road, 2026-07-21T16:11:42Z, 103
FBY, 073, Eastbound - Platform 2, Edgwar

Questions to answer:

- Does the train frequency at Fulham Broadway give a good estimate of average waiting time at Southfields?
- What does a normal service pattern look like? 
- Can disruption be detected early?

Data collection process is shown below (not reproducible). To repeat, make a .env file with API key, then run all cells.

In [ ]:
import time
import pandas as pd
from datetime import datetime, timezone

eb_trains = pd.DataFrame(columns=["station", "train_no", "destination", "expected_arr", "time_to_station", "tfl_status", "time"])
wb_trains = pd.DataFrame(columns=["station", "train_no", "destination", "expected_arr", "time_to_station", "tfl_status", "time"])

for i in range(180):
    # tube status
    status = get_line_status("district")[0]["lineStatuses"][0]["statusSeverityDescription"]

    # train arrivals EB
    for stop in ["940GZZLUWIP", "940GZZLUSFS", "940GZZLUEPY", "940GZZLUPYB", "940GZZLUPSG", "940GZZLUFBY"]:  # Wimbledon Park to Fulham Broadway
        arrivals = find_arrivals("district", stop, direction="outbound")
        for tube in arrivals:
            eb_trains.loc[len(eb_trains)] = {
                "station": tube["naptanId"][-3:],
                "train_no": tube["vehicleId"],
                "destination": tube["towards"],
                "expected_arr": tube["expectedArrival"],
                "time_to_station": tube["timeToStation"],
                "tfl_status": status,
                "time": str(datetime.now(timezone.utc)),
            }

    # same WB
    for stop in ["940GZZLUWIP", "940GZZLUSFS", "940GZZLUEPY", "940GZZLUPYB", "940GZZLUPSG", "940GZZLUFBY"]:  # Wimbledon Park to Fulham Broadway
        arrivals = find_arrivals("district", stop, direction="inbound")
        for tube in arrivals:
            wb_trains.loc[len(wb_trains)] = {
                "station": tube["naptanId"][-3:],
                "train_no": tube["vehicleId"],
                "destination": tube["towards"],
                "expected_arr": tube["expectedArrival"],
                "time_to_station": tube["timeToStation"],
                "tfl_status": status,
                "time": str(datetime.now(timezone.utc)),
            }
        
    time.sleep(20)

eb_trains.to_csv("eb_trains.csv", index=False)
wb_trains.to_csv("wb_trains.csv", index=False)